In [1]:
# ============================================================
# Qwen2.5-VL-7B question-aware caching -- merged version
#
# Combines Ponish's and Rohan's independent drafts. Two real fixes
# over Rohan's version, worth knowing about:
#   1. EMPTY_IDS is rebuilt per-image (see cell 9), not computed once
#      globally at a fixed 336x336 -- Qwen's dynamic resolution means
#      a fixed-size empty template would misalign against real images
#      of a different size, breaking question_span silently.
#   2. Cells 6 and 7 are kept SEPARATE (HALLUC_TYPE/USERNAME config,
#      then the VG download), matching what the runbook's prose says
#      ("cell 6 sets HALLUC_TYPE", "cell 7 triggers the VG download").
#      Rohan's actual uploaded notebook merges these into one cell,
#      which contradicts his own runbook -- worth flagging back to him.
# ============================================================

!pip install -q -U transformers accelerate qwen-vl-utils requests

import os
import json
import glob
import random
import time
import torch
import torch.nn.functional as F
from PIL import Image
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
print("torch:", torch.__version__)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 88.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 78.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 53.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have

In [2]:
MODEL_NAME = "Qwen/Qwen2.5-VL-7B-Instruct"

# min_pixels/max_pixels bound how far Qwen's dynamic resolution can
# range, so a single unusually large image can't blow up the image
# token count (and memory) unpredictably. From Rohan's draft.
processor = AutoProcessor.from_pretrained(
    MODEL_NAME,
    min_pixels=256 * 28 * 28,
    max_pixels=1024 * 28 * 28,
)
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    max_memory={0: "12GiB", 1: "12GiB"},
)
model.eval()
for p in model.parameters():
    p.requires_grad_(False)

print("model loaded:", MODEL_NAME)


preprocessor_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/216 [00:00<?, ?B/s]

model loaded: Qwen/Qwen2.5-VL-7B-Instruct


In [3]:
# ============================================================
# DIAGNOSTIC CELL -- READ THE FULL OUTPUT BEFORE RUNNING CELL 3.
# This only inspects the model. It changes nothing.
# ============================================================

print("=" * 70)
print("TOP-LEVEL MODEL STRUCTURE")
print("=" * 70)
print(model)

print()
print("=" * 70)
print("LANGUAGE MODEL PATH CHECK")
print("=" * 70)
lm = None
lm_path = None
for path in ["model.model.language_model", "model.language_model"]:
    try:
        obj = eval(path)
        print(f"{path} exists -- type: {type(obj)}")
        lm = obj
        lm_path = path
    except AttributeError:
        print(f"{path} -- NOT FOUND")

if lm is not None and hasattr(lm, "layers"):
    print(f"num layers (len of .layers): {len(lm.layers)}")
else:
    print("could not find .layers on the language model -- inspect the structure above manually")

if lm is not None:
    print(f"language_model top-level modules: {[name for name, _ in lm.named_children()]}")
    print("-> confirms whether a submodule called 'norm' exists at this path")

print()
print("=" * 70)
print("VISION MODULE CHECK")
print("=" * 70)
print("model.visual exists:", hasattr(model, "visual"))
if hasattr(model, "visual"):
    print("model.visual top-level modules:", [name for name, _ in model.visual.named_children()])

print()
print("=" * 70)
print("CONFIG FIELDS")
print("=" * 70)
print("config.image_token_id =", getattr(model.config, "image_token_id", "NOT FOUND"))
print("config.vision_start_token_id =", getattr(model.config, "vision_start_token_id", "NOT FOUND"))
print("config.vision_end_token_id =", getattr(model.config, "vision_end_token_id", "NOT FOUND"))
_text_cfg = getattr(model.config, "text_config", None)
print("config.text_config.hidden_size =", getattr(_text_cfg, "hidden_size", "NOT FOUND"))
print("config.text_config.num_hidden_layers =", getattr(_text_cfg, "num_hidden_layers", "NOT FOUND"))
print("config.hidden_size (top-level fallback) =", getattr(model.config, "hidden_size", "NOT FOUND"))

print()
print("If anything above says NOT FOUND, search the full config dump below")
print("for fields containing 'image' or 'vision', and use whatever you find")
print("in cell 3 instead of the assumed path.")
print()
print("=" * 70)
print("FULL CONFIG DUMP")
print("=" * 70)
print(model.config)


TOP-LEVEL MODEL STRUCTURE
Qwen2_5_VLForConditionalGeneration(
  (model): Qwen2_5_VLModel(
    (visual): Qwen2_5_VisionTransformerPretrainedModel(
      (patch_embed): Qwen2_5_VisionPatchEmbed(
        (proj): Conv3d(3, 1280, kernel_size=(2, 14, 14), stride=(2, 14, 14), bias=False)
      )
      (rotary_pos_emb): Qwen2_5_VLVisionRotaryEmbedding()
      (blocks): ModuleList(
        (0-31): 32 x Qwen2_5_VLVisionBlock(
          (norm1): Qwen2_5_VLRMSNorm((1280,), eps=1e-06)
          (norm2): Qwen2_5_VLRMSNorm((1280,), eps=1e-06)
          (attn): Qwen2_5_VLVisionAttention(
            (qkv): Linear(in_features=1280, out_features=3840, bias=True)
            (proj): Linear(in_features=1280, out_features=1280, bias=True)
          )
          (mlp): Qwen2_5_VLMLP(
            (gate_proj): Linear(in_features=1280, out_features=3420, bias=True)
            (up_proj): Linear(in_features=1280, out_features=3420, bias=True)
            (down_proj): Linear(in_features=3420, out_features=1280, b

In [4]:
# ============================================================
# If cell 2's output didn't match these assumptions, EDIT THIS
# CELL before running it. It will throw a clear AttributeError
# if a path is wrong -- that is the intended, safe failure mode,
# not something to work around silently.
# ============================================================

ln_f = model.model.language_model.norm
lm_head = model.lm_head

n_layer = len(model.model.language_model.layers)

# defensive fallback for hidden_dim -- some HF config layouts nest
# this under text_config, others put it at the top level
hidden_dim = (
    model.config.text_config.hidden_size
    if hasattr(model.config, "text_config")
    else model.config.hidden_size
)

image_token_id = model.config.image_token_id

print(f"n_layer = {n_layer}")
print(f"hidden_dim = {hidden_dim}")
print(f"image_token_id = {image_token_id}")

# Vision hook -- registered on the whole vision module (model.visual).
# For Qwen2.5-VL this includes the patch-merger, so this captures
# POST-merger embeddings already living in hidden_dim -- NOT the same
# thing as LLaVA's pre-projector CLIP hook. There is also no CLS token
# to strip here (Qwen's ViT never prepends one).
#
# The exact output type is unverified without a real forward pass, so
# this handles every plausible wrapping (a tuple, a HF-style output
# object with .last_hidden_state, or a raw tensor) rather than
# assuming one -- and the pooling step in cell 9 checks dimensionality
# too, so a wrong guess here fails loudly instead of silently.
_encoder_cache = {}

def _vision_tower_hook(module, inputs, output):
    if isinstance(output, tuple):
        out = output[0]
    elif hasattr(output, "last_hidden_state"):
        out = output.last_hidden_state
    else:
        out = output
    _encoder_cache["patch_embeddings"] = out.detach().cpu()

_hook_handle = model.model.visual.register_forward_hook(_vision_tower_hook)
print("hook registered on model.visual")


n_layer = 28
hidden_dim = 3584
image_token_id = 151655
hook registered on model.visual


In [5]:
def find_file(base, filename):
    for root, _, files in os.walk(base):
        if filename in files:
            return os.path.join(root, filename)
    return None

COCO_ROOT = "/kaggle/input/datasets/nadaibrahim/coco2014"

# try the known layout first (fast), fall back to a search if the
# dataset happens to be mounted with different nesting
_known_coco_path = os.path.join(COCO_ROOT, "val2014", "val2014")
if os.path.isdir(_known_coco_path):
    img_dir = _known_coco_path
else:
    sample_path = find_file(COCO_ROOT, "COCO_val2014_000000000139.jpg")
    assert sample_path is not None, f"couldn't find a sample COCO file under {COCO_ROOT} -- check COCO_ROOT"
    img_dir = os.path.dirname(sample_path)
print("COCO images live in:", img_dir)

AMBER_IMAGE_DIR = "/kaggle/input/datasets/nocturnalnerd18/amber-hallucination/image"


def load_image(path):
    return Image.open(path).convert("RGB")


def build_inputs(image, text):
    conversation = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": text}]}]
    prompt_text = processor.apply_chat_template(conversation, add_generation_prompt=True)
    return processor(text=prompt_text, images=image, return_tensors="pt").to(model.device, torch.float16)


COCO images live in: /kaggle/input/datasets/nadaibrahim/coco2014/val2014/val2014


In [6]:
VG_IMAGE_DIRS = ["/kaggle/temp", "/kaggle/temp/VG_100K_2"]

def find_vg_image_path(image_id: str) -> str:
    for d in VG_IMAGE_DIRS:
        candidate = os.path.join(d, f"{image_id}.jpg")
        if os.path.exists(candidate):
            return candidate
    return None


def resolve_image_path(q: dict) -> str:
    qid = q["question_id"]
    if qid.startswith("pope_"):
        return os.path.join(img_dir, f"{q['image_id']}.jpg")
    elif qid.startswith("amber_"):
        return os.path.join(AMBER_IMAGE_DIR, f"{q['image_id']}.jpg")
    elif qid.startswith("reefknot_"):
        return find_vg_image_path(q["image_id"])
    return None


In [7]:
HALLUC_TYPE = "relation"   # switched from attribute -- this run triggers the Visual Genome download in cell 7
USERNAME = "ponishbhatia"   # <-- already set

QA_CACHE_DIR = f"/kaggle/working/cache/question_aware_qwen25vl_{HALLUC_TYPE}"
os.makedirs(QA_CACHE_DIR, exist_ok=True)
print(f"HALLUC_TYPE = {HALLUC_TYPE!r}")
print(f"output dir  = {QA_CACHE_DIR}")


HALLUC_TYPE = 'relation'
output dir  = /kaggle/working/cache/question_aware_qwen25vl_relation


In [8]:
if HALLUC_TYPE == "relation":
    !wget https://cs.stanford.edu/people/rak248/VG_100K/images.zip -O /kaggle/working/VG_100K.zip
    !wget https://cs.stanford.edu/people/rak248/VG_100K_2/images2.zip -O /kaggle/working/VG_100K_2.zip
    !unzip -q -o /kaggle/working/VG_100K.zip -d /kaggle/temp/
    !unzip -q -o /kaggle/working/VG_100K_2.zip -d /kaggle/temp/
    for zpath in ["/kaggle/working/VG_100K.zip", "/kaggle/working/VG_100K_2.zip"]:
        if os.path.exists(zpath):
            os.remove(zpath)
    print("Visual Genome downloaded, extracted, zips removed")
else:
    print(f"HALLUC_TYPE={HALLUC_TYPE!r} -- Visual Genome not needed, skipping download")


--2026-09-29 08:25:45--  https://cs.stanford.edu/people/rak248/VG_100K/images.zip
Resolving cs.stanford.edu (cs.stanford.edu)... 171.64.64.64
Connecting to cs.stanford.edu (cs.stanford.edu)|171.64.64.64|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 9730308001 (9.1G) [application/zip]
Saving to: ‘/kaggle/working/VG_100K.zip’

/kaggle/working/VG_ 100%[===================>]   9.06G  24.0MB/s    in 8m 34s  

2026-09-29 08:34:19 (18.1 MB/s) - ‘/kaggle/working/VG_100K.zip’ saved [9730308001/9730308001]

--2026-09-29 08:34:19--  https://cs.stanford.edu/people/rak248/VG_100K_2/images2.zip
Resolving cs.stanford.edu (cs.stanford.edu)... 171.64.64.64
Connecting to cs.stanford.edu (cs.stanford.edu)|171.64.64.64|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 5471658058 (5.1G) [application/zip]
Saving to: ‘/kaggle/working/VG_100K_2.zip’

/kaggle/working/VG_ 100%[===================>]   5.10G  33.5MB/s    in 6m 18s  

2026-09-29 08:40:38 (13.8 MB

In [9]:
QUESTIONS_PATH = "/kaggle/input/datasets/nocturnalnerd18/vlm-questions/all_questions.json"
with open(QUESTIONS_PATH) as f:
    all_questions = json.load(f)

assert len(all_questions) == 17492, "attach vlm-questions version 2"

questions_subset = [
    q for q in all_questions
    if q["hallucination_type"] == HALLUC_TYPE and q["answer_format"] == "yes_no"
]
print(f"{len(questions_subset)} {HALLUC_TYPE} yes/no questions selected")


6506 relation yes/no questions selected


In [10]:
def get_empty_ids(image):
    """
    Qwen2.5-VL uses dynamic resolution -- the number of image tokens
    depends on the image's own dimensions, not a fixed constant like
    LLaVA's 576. So EMPTY_IDS can NOT be a single global constant
    computed once at a fixed size (e.g. 336x336, LLaVA's convention) --
    it has to be rebuilt per image SIZE, or the diff in question_span
    misaligns against real images of a different resolution and finds
    the wrong span. A blank image of the same width/height should
    tokenize to the same image-token count as the real image, since
    that count is determined by resolution, not content -- this
    assumption is unverified until you actually see it hold in cell 10.
    """
    blank = Image.new("RGB", image.size)
    return build_inputs(blank, text="")["input_ids"][0].tolist()


def question_span(ids, empty_ids):
    n = min(len(ids), len(empty_ids))
    start = next(i for i in range(n) if ids[i] != empty_ids[i])
    tail = next(i for i in range(n) if ids[-1 - i] != empty_ids[-1 - i])
    return start, len(ids) - tail


@torch.no_grad()
def cache_question_aware(image_path, image_id, question_id, question_text):
    image = load_image(image_path)
    inputs = build_inputs(image, text=question_text)
    ids = inputs["input_ids"][0].tolist()

    empty_ids = get_empty_ids(image)
    q_start, q_end = question_span(ids, empty_ids)

    _encoder_cache.clear()
    outputs = model(**inputs, output_hidden_states=True)
    last_logits = outputs.logits[0, -1]

    image_positions = (inputs["input_ids"][0] == image_token_id).nonzero(as_tuple=True)[0]

    p1_hidden = torch.stack([layer[0, -1, :].half().cpu() for layer in outputs.hidden_states])
    p2_hidden = torch.stack([layer[0, q_start:q_end, :].mean(dim=0).half().cpu() for layer in outputs.hidden_states])
    pooled_image_hidden = torch.stack([layer[0, image_positions, :].mean(dim=0).half().cpu() for layer in outputs.hidden_states])

    # Defensive pooling for the vision-encoder hook output -- shape is
    # unverified (see cell 3), so handle both a batched (1, N, hidden)
    # and an unbatched (N, hidden) result rather than assuming either.
    enc = _encoder_cache["patch_embeddings"]
    if enc.dim() == 3:
        pooled_encoder_embedding = enc.mean(dim=1)[0].half()
    elif enc.dim() == 2:
        pooled_encoder_embedding = enc.mean(dim=0).half()
    else:
        raise RuntimeError(f"unexpected vision hook output shape: {tuple(enc.shape)}")

    return {
        "image_id": image_id, "question_id": question_id, "question_text": question_text,
        "answer_text": processor.tokenizer.decode(last_logits.argmax().item()).strip(),
        "span_text": processor.tokenizer.decode(ids[q_start:q_end]),
        "n_image_tokens": int(image_positions.numel()),
        "p1_hidden": p1_hidden,
        "p2_hidden": p2_hidden,
        "pooled_image_hidden": pooled_image_hidden,
        "pooled_encoder_embedding": pooled_encoder_embedding,
        "answer_logits": last_logits.half().cpu(),
    }


In [11]:
# ============================================================
# VALIDATION CHECKPOINT -- read all of it before running cell 11.
# There is no old Qwen cache to regress against (this is the first
# run), so these checks establish ground truth for THIS model
# instead of comparing against a prior pipeline.
#
# Uses a RANDOM sample (not just the first 50 in the file), since the
# question bank is concatenated by source (AMBER then Reefknot), so
# the first 50 in order would under-represent one source. Missing
# images are skipped, not allowed to crash the whole checkpoint.
# ============================================================

random.seed(42)
val_subset = random.sample(questions_subset, min(50, len(questions_subset)))

t0 = time.time()
val_entries = []
val_qs = []
for q in val_subset:
    image_path = resolve_image_path(q)
    if image_path is None or not os.path.exists(image_path):
        continue
    val_entries.append(cache_question_aware(image_path, q["image_id"], q["question_id"], q["question_text"]))
    val_qs.append(q)
sec_per_question = (time.time() - t0) / len(val_entries)

print(f"{len(val_entries)} questions validated, {sec_per_question:.2f} s/question")
print()

# --- Check 1: which normalization convention does this model's cached
# final-layer hidden state follow? Exactly one of these two should be
# near zero; the other should be clearly large. ---
raw_err = max(
    (lm_head(e["p1_hidden"][-1].to(lm_head.weight.device)).float().cpu() - e["answer_logits"].float()).abs().max().item()
    for e in val_entries
)
normed_err = max(
    (lm_head(ln_f(e["p1_hidden"][-1].to(lm_head.weight.device))).float().cpu() - e["answer_logits"].float()).abs().max().item()
    for e in val_entries
)

print("--- normalization convention check ---")
print(f"lm_head(p1_hidden[-1]) directly vs real logits, max abs err:    {raw_err:.4f}")
print(f"lm_head(ln_f(p1_hidden[-1])) vs real logits, max abs err:       {normed_err:.4f}")
if raw_err < 0.1 and normed_err >= 0.1:
    print(">>> convention: hidden_states[-1] is ALREADY post-final-norm (same as LLaVA/Vicuna).")
    print(">>> No extra ln_f needed when decoding from the last layer downstream.")
elif normed_err < 0.1 and raw_err >= 0.1:
    print(">>> convention: hidden_states[-1] is PRE-final-norm.")
    print(">>> Any downstream logit-lens/tuned-lens code MUST apply ln_f before lm_head.")
else:
    print(">>> NEITHER is near zero (or both are) -- something upstream is wrong.")
    print(">>> Stop and recheck the attribute paths in cell 3 / diagnostic in cell 2.")
print()

# --- Check 2: does the very last prompt token (right before the answer)
# stay fixed across different questions? ---
layer0_identical = all(torch.equal(e["p1_hidden"][0], val_entries[0]["p1_hidden"][0]) for e in val_entries)
print(f"layer-0 p1 identical across questions: {layer0_identical}")
if not layer0_identical:
    print(">>> the prompt template isn't ending the same way every time -- investigate before trusting anything downstream.")

# --- Check 3: does the found question span decode back to the real question? ---
spans_match = sum(e["span_text"].strip() == q["question_text"].strip() for e, q in zip(val_entries, val_qs))
print(f"question span decodes to the question: {spans_match}/{len(val_entries)}")

# --- Check 4: informational -- confirms the image-token count really
# does vary, and gives a sense of the range for memory planning. ---
n_img_tokens = [e["n_image_tokens"] for e in val_entries]
print(f"image token count range: {min(n_img_tokens)} - {max(n_img_tokens)}")
print()

# --- Check 5: the one check that's on you, not the code. ---
print("--- sample answers (read these yourself -- do they look like a real model answering?) ---")
for e in val_entries[:15]:
    print(f"  {e['question_id']:20s} {e['question_text'][:50]:50s} -> {e['answer_text']!r}")

print()
print("Don't proceed to cell 11 until all of the above looks right.")


50 questions validated, 0.80 s/question

--- normalization convention check ---
lm_head(p1_hidden[-1]) directly vs real logits, max abs err:    0.0156
lm_head(ln_f(p1_hidden[-1])) vs real logits, max abs err:       13.3867
>>> convention: hidden_states[-1] is ALREADY post-final-norm (same as LLaVA/Vicuna).
>>> No extra ln_f needed when decoding from the last layer downstream.

layer-0 p1 identical across questions: True
question span decodes to the question: 50/50
image token count range: 260 - 1008

--- sample answers (read these yourself -- do they look like a real model answering?) ---
  reefknot_2349891_0   Is the man taking picture in this photo? Please an -> 'no'
  amber_14469          Is there direct contact between the bird and beach -> 'Yes'
  amber_13761          Is there direct contact between the child and gras -> 'No'
  reefknot_2415389_0   Is the man teaching boy in this photo? Please answ -> 'yes'
  reefknot_2332110_1   Is the tire under the taxi in this photo? Please a 

In [12]:
import subprocess

UPLOAD_EVERY = 250
BATCH_SAVE_SIZE = UPLOAD_EVERY   # every 250 cached questions = 1 new file = 1 upload
FORCE_FRESH = False              # True only if you've checked your Kaggle profile and the dataset really isn't there

assert USERNAME and not USERNAME.startswith("REPLACE"), "set USERNAME in cell 6"
dataset_slug = f"question-aware-cache-qwen25vl-{HALLUC_TYPE}"


def _run(cmd):
    r = subprocess.run(cmd, capture_output=True, text=True)
    out = (r.stdout + r.stderr).strip()
    ok = r.returncode == 0 and "error" not in out.lower()   # the CLI sometimes prints errors but exits 0
    return ok, out


# ---- 1. restore anything already saved in the Kaggle dataset ----------------
ok, out = _run(["kaggle", "datasets", "download", "-d", f"{USERNAME}/{dataset_slug}",
                "-p", QA_CACHE_DIR, "--unzip"])
if ok:
    print("restored existing cache files from the Kaggle dataset")
else:
    low = out.lower()
    looks_missing = any(s in low for s in ("404", "403", "not found", "does not exist", "forbidden"))
    if not looks_missing and not FORCE_FRESH:
        raise RuntimeError("download failed for an unrelated reason, refusing to start fresh:\n" + out)
    if not FORCE_FRESH:
        # Kaggle answers 403 for both "doesn't exist" and "no access": check your own dataset list
        lok, lout = _run(["kaggle", "datasets", "list", "-m", "-s", dataset_slug, "-v"])
        if not lok:
            raise RuntimeError("can't verify your Kaggle login/credentials. If you're sure nothing is "
                               "uploaded yet, set FORCE_FRESH = True and re-run:\n" + lout)
        if f"{USERNAME}/{dataset_slug}" in lout:
            raise RuntimeError("dataset exists but the download was forbidden:\n" + out)
    print("no existing dataset on Kaggle yet, starting fresh")

with open(os.path.join(QA_CACHE_DIR, "dataset-metadata.json"), "w") as f:
    json.dump({"title": dataset_slug, "id": f"{USERNAME}/{dataset_slug}",
               "licenses": [{"name": "CC0-1.0"}]}, f)

dataset_exists = ok   # True if we just restored one, so the next upload is a "version"


def upload_to_kaggle(msg):
    """Upload the whole folder. A failure warns and continues; the files go up with the next upload."""
    global dataset_exists
    t0 = time.time()
    create_cmd = ["kaggle", "datasets", "create", "-p", QA_CACHE_DIR]
    version_cmd = ["kaggle", "datasets", "version", "-p", QA_CACHE_DIR, "-m", msg]
    if not dataset_exists:
        ok_, out_ = _run(create_cmd)
        if not ok_:
            ok_, out2 = _run(version_cmd)
            out_ += "\n" + out2
    else:
        ok_, out_ = _run(version_cmd)
    if ok_:
        dataset_exists = True
        print(f"    [upload ok in {time.time()-t0:.0f}s] {msg}")
    else:
        print(f"    [UPLOAD FAILED, continuing] {out_[-400:]}")
    return ok_


# ---- 2. resume: skip every question already saved ---------------------------
existing_batches = glob.glob(os.path.join(QA_CACHE_DIR, f"{HALLUC_TYPE}_batch_*.pt"))
already_done_ids = set()
for fpath in existing_batches:
    already_done_ids.update(e["question_id"] for e in torch.load(fpath))
print(f"resuming: {len(already_done_ids)} questions already saved in {len(existing_batches)} files")

batch_index = len(existing_batches)
batch_entries, skipped, span_mismatch = [], 0, 0
t_start = time.time()

# ---- 3. main loop ------------------------------------------------------------
for i, q in enumerate(questions_subset):
    if q["question_id"] in already_done_ids:
        continue

    image_path = resolve_image_path(q)
    if image_path is None or not os.path.exists(image_path):
        skipped += 1
        continue

    entry = cache_question_aware(image_path, q["image_id"], q["question_id"], q["question_text"])
    entry.update({k: q[k] for k in ("ground_truth_answer", "answer_format", "hallucination_type")})
    span_mismatch += entry["span_text"].strip() != q["question_text"].strip()
    batch_entries.append(entry)

    if len(batch_entries) >= BATCH_SAVE_SIZE:
        torch.save(batch_entries, os.path.join(QA_CACHE_DIR, f"{HALLUC_TYPE}_batch_{batch_index}.pt"))
        batch_index += 1
        batch_entries = []
        upload_to_kaggle(f"{HALLUC_TYPE}: {batch_index} batches, through question {i+1}/{len(questions_subset)}")

    if i % 10 == 0 and i > 0:
        elapsed = time.time() - t_start
        avg = elapsed / (i + 1)
        remaining = avg * (len(questions_subset) - i - 1)
        print(f"[{i+1}/{len(questions_subset)}] {q['question_id']}: {entry['answer_text']!r} "
              f"({avg:.2f}s/question avg, ETA {remaining/60:.1f} min, skipped: {skipped}, "
              f"span mismatches: {span_mismatch}, img_tokens: {entry['n_image_tokens']})")

if batch_entries:
    torch.save(batch_entries, os.path.join(QA_CACHE_DIR, f"{HALLUC_TYPE}_batch_{batch_index}.pt"))
    batch_index += 1
upload_to_kaggle(f"{HALLUC_TYPE}: FINAL, {batch_index} batches")

print(f"done -- {batch_index} files, {skipped} skipped, {span_mismatch} span mismatches, "
      f"{(time.time()-t_start)/60:.1f} min this session")

no existing dataset on Kaggle yet, starting fresh
resuming: 0 questions already saved in 0 files
[11/6506] amber_13567: 'No' (0.56s/question avg, ETA 60.1 min, skipped: 0, span mismatches: 0, img_tokens: 280)
[21/6506] amber_13577: 'No' (0.55s/question avg, ETA 59.7 min, skipped: 0, span mismatches: 0, img_tokens: 288)
[31/6506] amber_13587: 'No' (0.55s/question avg, ETA 59.2 min, skipped: 0, span mismatches: 0, img_tokens: 266)
[41/6506] amber_13597: 'No' (0.55s/question avg, ETA 58.9 min, skipped: 0, span mismatches: 0, img_tokens: 266)
[51/6506] amber_13607: 'No' (0.55s/question avg, ETA 58.7 min, skipped: 0, span mismatches: 0, img_tokens: 270)
[61/6506] amber_13617: 'No' (0.55s/question avg, ETA 58.6 min, skipped: 0, span mismatches: 0, img_tokens: 266)
[71/6506] amber_13627: 'No' (0.55s/question avg, ETA 58.5 min, skipped: 0, span mismatches: 0, img_tokens: 280)
[81/6506] amber_13637: 'No' (0.55s/question avg, ETA 58.5 min, skipped: 0, span mismatches: 0, img_tokens: 266)
[91/650

In [13]:
non_yesno = sum(1 for f in glob.glob(os.path.join(QA_CACHE_DIR, "*.pt"))
                 for e in torch.load(f)
                 if e["answer_text"].lower() not in ("yes", "no"))
print(f"{non_yesno} non-yes/no answers out of {len(questions_subset)}")

93 non-yes/no answers out of 6506


In [14]:
if skipped:
    print(f"WARNING: {skipped} questions were skipped this session (missing images), so the dataset may be incomplete")

ok, out = _run(["kaggle", "datasets", "files", f"{USERNAME}/{dataset_slug}"])
print(out)
print()
print(f"published: {USERNAME}/{dataset_slug}")
print("For Rohan: send back this dataset link, the full printed output of")
print("cell 10 (the normalization-convention result matters most), and")
print("total GPU time used for this run.")

Next Page Token = CfDJ8LXGgraiXMNClPzcySXY5U6tzoGW9djmrnHNgenxyS-O1lM_lIlQ9Zw2PlefQV-160mIsquG-PqzCAehZ7BkWxxiVbZDeaYOhk7SnQJmnzuuugOFwcv3mBuvt97PkZJ7B1RQeOp4kd82Lz8ksQ
name                       size  creationDate                
--------------------  ---------  --------------------------  
relation_batch_0.pt   232989917  2026-09-29 10:16:54.741000  
relation_batch_1.pt   232990365  2026-09-29 10:16:55.238000  
relation_batch_10.pt  233000389  2026-09-29 10:16:54.159000  
relation_batch_11.pt  233000005  2026-09-29 10:16:53.617000  
relation_batch_12.pt  233000645  2026-09-29 10:16:53.923000  
relation_batch_13.pt  233000453  2026-09-29 10:16:53.970000  
relation_batch_14.pt  233000773  2026-09-29 10:16:55.438000  
relation_batch_15.pt  233000261  2026-09-29 10:16:54.231000  
relation_batch_16.pt  233000261  2026-09-29 10:16:55.318000  
relation_batch_17.pt  233000965  2026-09-29 10:16:55.019000  
relation_batch_18.pt  233000773  2026-09-29 10:16:54.179000  
relation_batch_19.pt  233